# Duplicate Detection
Finds exact duplicate rows and near-duplicates based on key columns.

**Configure the cell below**, then **Run All** to analyze. The final cell writes a deduped table — run it manually.

In [ ]:
# ── Configuration ──────────────────────────────────────────────
TABLE_NAME = "{{TABLE_NAME}}"
LAKEHOUSE_NAME = "{{LAKEHOUSE_NAME}}"
# Columns that form the natural key for near-duplicate detection
# Set to [] to skip near-duplicate check
KEY_COLUMNS = {{KEY_COLUMNS}}  # e.g., ["id", "name", "date"]
OUTPUT_SUFFIX = "_cleaned"
# ───────────────────────────────────────────────────────────────

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder.getOrCreate()
# Read from _cleaned if it exists (previous fixes), else original
try:
    df = spark.table(f"{TABLE_NAME}{OUTPUT_SUFFIX}")
    print(f"Reading from {TABLE_NAME}{OUTPUT_SUFFIX} (previous fixes applied)")
except:
    df = spark.table(TABLE_NAME)
    print(f"Reading from {TABLE_NAME} (original table)")
original_count = df.count()

print(f"Table: {TABLE_NAME}")
print(f"Rows: {original_count:,}")

In [ ]:
print("=" * 60)
print("EXACT DUPLICATE ANALYSIS")
print("=" * 60)

all_cols = df.columns
w = Window.partitionBy(all_cols)
df_with_dup_count = df.withColumn("_dup_count", F.count("*").over(w))
exact_dups = df_with_dup_count.where(F.col("_dup_count") > 1).drop("_dup_count")
exact_dup_count = exact_dups.count()

print(f"\nExact duplicate rows: {exact_dup_count:,}")
if exact_dup_count > 0:
    print("Sample duplicates:")
    display(exact_dups.limit(10))
else:
    print("No exact duplicates found.")

In [ ]:
print("=" * 60)
print("NEAR-DUPLICATE ANALYSIS")
print("=" * 60)

if KEY_COLUMNS:
    w_key = Window.partitionBy(KEY_COLUMNS)
    df_with_key_dup = df.withColumn("_key_dup_count", F.count("*").over(w_key))
    near_dups = df_with_key_dup.where(F.col("_key_dup_count") > 1).drop("_key_dup_count")
    near_dup_count = near_dups.count()

    print(f"\nNear-duplicates (same {KEY_COLUMNS}): {near_dup_count:,}")
    if near_dup_count > 0:
        print("Sample near-duplicates:")
        display(near_dups.orderBy(KEY_COLUMNS).limit(20))
else:
    print("\nNo key columns specified — skipping near-duplicate detection.")
    print("Tip: Set KEY_COLUMNS in the config cell to check for duplicates on specific columns.")

---
## ⚠ Apply Fix — Remove Exact Duplicates
The cell below removes exact duplicate rows (keeps first occurrence) and writes to a new table.

**Review the analysis above before running this cell.**

In [ ]:
# ⚠ APPLY FIX — Run manually after review
cleaned_df = df.dropDuplicates()
final_count = cleaned_df.count()
removed = original_count - final_count

output_table = f"{TABLE_NAME}{OUTPUT_SUFFIX}"
cleaned_df.write.mode("overwrite").format("delta").saveAsTable(
    output_table
)

print(f"Removed {removed:,} exact duplicate rows")
print(f"Output: {LAKEHOUSE_NAME}.{output_table} ({final_count:,} rows)")